In [ ]:
from dotenv import load_dotenv
load_dotenv()



True

In [2]:
import yaml

# Load agents from YAML file
with open('agents.yaml', 'r') as file:
    agent_config = yaml.safe_load(file)

# Load tasks from YAML file
with open('multiple_tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

In [5]:
# pip install crewai-tools
from crewai.tools import tool

In [6]:
@tool("retreive_return_policy",)
def retreive_return_policy(query:str):
     """
    Returns the company's return policy.

    Args:
        query (str): Customer's inquiry about the return policy as a python string
    """
     return (
        "Our return policy states that if the item is not damaged, "
        "you have a receipt of purchase, and you bought it within the last 60 days, "
        "you can return it and receive store credit."
    )


@tool("retrieve_shipping_policy")
def retrieve_shipping_policy() -> str:
    """
    description: Returns the company's shipping policy for handling returns.
    """
    return (
        "Our shipping policy for returns is as follows: "
        "Processing and delivery of returned items typically take 5-7 business days. "
        "Customers are responsible for a flat return shipping fee of $5 unless the item is defective. "
        "Tracking numbers are provided once the return is processed and shipped."
    )



@tool("retrieve_complaint_protocol")
def retrieve_complaint_protocol() -> str:
    """
    description: Provides the company's complaint resolution process for return-related issues.
    """
    return (
        "Our complaint resolution process for return-related issues is as follows: "
        "Complaints are acknowledged within 24 hours of submission. "
        "The resolution steps include: "
        "1) Reviewing the complaint and validating the return case, "
        "2) Contacting the customer for clarification or additional details if needed, and "
        "3) Providing a resolution, such as a refund, replacement, or escalation. "
        "If unresolved within 3 business days, the case is escalated to a senior support representative for priority handling."
    )

In [7]:
from crewai import Agent, Task, Crew, Process

In [11]:
agent_details = agent_config["customer_support_agent"]

customer_support_agent = Agent(
    role=agent_details["role"],
    goal=agent_details["goal"],
    backstory=agent_details["backstory"],
    tools=[retreive_return_policy, retrieve_shipping_policy, retrieve_complaint_protocol],
    verbose=True,
    llm = "gpt-4o-mini"
)



# Task 1: Return Policy
return_task_config = task_config["return_task"]
return_task = Task(
    description=return_task_config["description"],
    expected_output=return_task_config["expected_output"],
    agent=customer_support_agent
)

# Task 2: Shipping Policy (depends on Return Policy)
shipping_task_config = task_config["shipping_task"]
shipping_task = Task(
    description=shipping_task_config["description"],
    expected_output=shipping_task_config["expected_output"],
    agent=customer_support_agent
)

# Task 3: Complaint Resolution (depends on Shipping Policy)
complaint_task_config = task_config["complaint_task"]
complaint_task = Task(
    description=complaint_task_config["description"],
    expected_output=complaint_task_config["expected_output"],
    agent=customer_support_agent
)

email_task_config = task_config["email_task"]
email_task = Task(
    description=email_task_config["description"],
    expected_output=email_task_config["expected_output"],
    agent=customer_support_agent
)

# Create and execute the crew
crew = Crew(
    agents=[customer_support_agent],
    tasks=[return_task, shipping_task, complaint_task, email_task],
    process=Process.sequential,
    memory=True
)


In [12]:
results = await crew.kickoff_async({"customer_inquiry": """I bought an item and I don't like it. 
                                    I want to return it. 
                                    What is your return policy for this case?"""})

╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: Customer Support Specialist                                          │
│                                                                              │
│  Task: Respond to the following customer inquiry: "I bought an item and I    │
│  don't like it. I want to return it. What is your return policy for this     │
│  case?"                                                                      │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Tool retreive_return_policy executed with result: Our return policy states that if the item is not damaged, you have a receipt of purchase, and you bought it within the last 60 days, you can

In [15]:
print(results.raw) #results.raw

Subject: Your Return Inquiry – Important Information Inside

Dear [Customer's Name],

Thank you for reaching out to us regarding your return inquiry. We value your satisfaction and are here to assist you.

Our **Return Policy** allows you to return items under the following conditions:

1. **Condition of the Item**: The item must be undamaged and in its original condition.
2. **Proof of Purchase**: You need to have the receipt or proof of purchase.
3. **Return Period**: The return must be initiated within 60 days from the date of purchase.

If you meet these criteria, you can return the item and receive store credit. In case the return does not meet the policy requirements, we will clearly explain the reasons and offer alternatives, such as exchanges.

**Shipping Policy for Returns**:
- The processing and delivery of returned items typically take 5-7 business days.
- Customers are responsible for a flat return shipping fee of $5 unless the item is defective.
- Tracking information will

In [16]:
agent_details = agent_config["customer_support_agent"]
customer_support_agent = Agent(
    role=agent_details["role"],
    goal=agent_details["goal"],
    backstory=agent_details["backstory"],
    tools=[],
    verbose=True,
)

In [17]:
return_task_config = task_config["return_task"]

return_task = Task(
    description=return_task_config["description"],
    expected_output=return_task_config["expected_output"],
    agent=customer_support_agent,
    tools=[retreive_return_policy]) 

shipping_task_config = task_config["shipping_task"]


shipping_task = Task(
    description=shipping_task_config["description"],
    expected_output=shipping_task_config["expected_output"],
    agent=customer_support_agent,
    tools=[retrieve_shipping_policy],
    context=[return_task]
)

complaint_task_config = task_config["complaint_task"]
complaint_task = Task(
    description=complaint_task_config["description"],
    expected_output=complaint_task_config["expected_output"],
    agent=customer_support_agent,
    tools=[retrieve_complaint_protocol],
    context=[return_task, shipping_task]
)


email_task_config = task_config["email_task"]
email_task = Task(
    description=email_task_config["description"],
    expected_output=email_task_config["expected_output"],
    agent=customer_support_agent,
    context=[return_task, shipping_task, complaint_task]
)


crew = Crew(
    agents=[customer_support_agent],
    tasks=[return_task, shipping_task, complaint_task, email_task],
    process=Process.sequential
)

In [18]:
results = await crew.kickoff_async({"customer_inquiry": "I bought an item and I don't like it. I want to return it. What is your return policy for this case?"})

╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: Customer Support Specialist                                          │
│                                                                              │
│  Task: Respond to the following customer inquiry: "I bought an item and I    │
│  don't like it. I want to return it. What is your return policy for this     │
│  case?"                                                                      │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Tool retreive_return_policy executed with result: Our return policy states that if the item is not damaged, you have a receipt of purchase, and you bought it within the last 60 days, you can